In [13]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight

# =========================
# 1) Load data (Kaggle)
# =========================
DATA_PATH = "/kaggle/input/employee-performance-and-productivity-data/Extended_Employee_Performance_and_Productivity_Data.csv"
df = pd.read_csv(DATA_PATH)

TARGET = "Resigned"
DROP_COLS = ["Employee_ID", "Hire_Date"]  # identifier/time columns

X = df.drop(columns=[TARGET] + [c for c in DROP_COLS if c in df.columns], errors="ignore")
y = df[TARGET].astype(int)

# Feature types
cat_cols = ["Department", "Gender", "Job_Title", "Education_Level"]
cat_cols = [c for c in cat_cols if c in X.columns]
num_cols = [c for c in X.columns if c not in cat_cols]

print("Data shape:", df.shape)
print("X shape:", X.shape, "| y mean (Resigned=1 rate):", float(y.mean()))
print("Categorical:", cat_cols)
print("Numeric:", num_cols)

# =========================
# 2) Train/Test split (stratified)
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain:", X_train.shape, "Test:", X_test.shape)
print("Train class distribution:\n", y_train.value_counts(normalize=True).rename("ratio"))

# =========================
# 3) Preprocess (OHE + passthrough)
# =========================
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)

# =========================
# 4) Handle imbalance WITHOUT imblearn:
#    sample_weight = balanced on TRAIN only
# =========================
train_sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

print("\nSample_weight summary:",
      "min=", float(train_sample_weight.min()),
      "max=", float(train_sample_weight.max()),
      "mean=", float(train_sample_weight.mean()))

# =========================
# 5) Models (Base + AdaBoost + Random Forest)
#    NOTE: RF'de class_weight=None bıraktık çünkü sample_weight veriyoruz.
# =========================
base_dt = DecisionTreeClassifier(max_depth=1, random_state=42)

ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
    n_estimators=300,
    learning_rate=0.5,
    random_state=42
)

rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    n_jobs=-1,
    class_weight=None,      # <-- balanced yerine None (sample_weight ile dengeleyeceğiz)
    min_samples_leaf=10
)

models = {
    "Base (DT stump)": base_dt,
    "AdaBoost (DT stump)": ada,
    "Random Forest": rf
}

# =========================
# 6) Train + Evaluate
# =========================
def eval_binary(pipe, X_te, y_te):
    y_pred = pipe.predict(X_te)

    # ROC-AUC için olasılık gerekli
    if hasattr(pipe, "predict_proba"):
        y_score = pipe.predict_proba(X_te)[:, 1]
    elif hasattr(pipe, "decision_function"):
        y_score = pipe.decision_function(X_te)
    else:
        y_score = None

    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_te, y_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y_te, y_pred, pos_label=1, zero_division=0)
    auc = roc_auc_score(y_te, y_score) if y_score is not None else np.nan

    cm = confusion_matrix(y_te, y_pred)
    return acc, prec, rec, f1, auc, cm

rows = []
cms = {}

for name, clf in models.items():
    pipe = Pipeline(steps=[("prep", preprocess), ("clf", clf)])

    # A ŞIKKI: RF DAHİL tüm modelleri sample_weight ile eğitiyoruz
    pipe.fit(X_train, y_train, clf__sample_weight=train_sample_weight)

    acc, prec, rec, f1, auc, cm = eval_binary(pipe, X_test, y_test)

    rows.append({
        "Model": name,
        "Accuracy": acc,
        "Precision(Resigned=1)": prec,
        "Recall(Resigned=1)": rec,
        "F1(Resigned=1)": f1,
        "ROC-AUC": auc
    })
    cms[name] = cm

results = pd.DataFrame(rows).sort_values(by="F1(Resigned=1)", ascending=False)

print("\n=== Bonus Ensemble Results (Resigned Classification) ===")
display(results)

print("\nConfusion matrices (rows=true [0,1], cols=pred [0,1])")
for k, cm in cms.items():
    print("\n", k)
    print(cm)

# =========================
# 7) Export LaTeX table (optional)
# =========================
tex_path = "bonus_ensemble_resigned_table.tex"
with open(tex_path, "w", encoding="utf-8") as f:
    f.write("\\begin{table}[h]\n\\centering\n")
    f.write("\\caption{Bonus ensemble comparison for Resigned prediction.}\n")
    f.write("\\label{tab:bonus_ensemble_resigned}\n")
    f.write("\\begin{tabular}{lccccc}\n\\toprule\n")
    f.write("Model & Acc. & Prec(1) & Rec(1) & F1(1) & AUC \\\\\n\\midrule\n")
    for _, r in results.iterrows():
        f.write(
            f"{r['Model']} & {r['Accuracy']:.5f} & {r['Precision(Resigned=1)']:.5f} & "
            f"{r['Recall(Resigned=1)']:.5f} & {r['F1(Resigned=1)']:.5f} & {r['ROC-AUC']:.5f} \\\\\n"
        )
    f.write("\\bottomrule\n\\end{tabular}\n\\end{table}\n")

print("\nWrote:", tex_path)


Data shape: (100000, 20)
X shape: (100000, 17) | y mean (Resigned=1 rate): 0.1001
Categorical: ['Department', 'Gender', 'Job_Title', 'Education_Level']
Numeric: ['Age', 'Years_At_Company', 'Performance_Score', 'Monthly_Salary', 'Work_Hours_Per_Week', 'Projects_Handled', 'Overtime_Hours', 'Sick_Days', 'Remote_Work_Frequency', 'Team_Size', 'Training_Hours', 'Promotions', 'Employee_Satisfaction_Score']

Train: (80000, 17) Test: (20000, 17)
Train class distribution:
 Resigned
0    0.8999
1    0.1001
Name: ratio, dtype: float64

Sample_weight summary: min= 0.55561729081009 max= 4.995004995004995 mean= 1.0

=== Bonus Ensemble Results (Resigned Classification) ===


,Model,Accuracy,Precision(Resigned=1),Recall(Resigned=1),F1(Resigned=1),ROC-AUC
0,Base (DT stump),0.25645,0.100118,0.804695,0.178080,0.500081
1,AdaBoost (DT stump),0.51400,0.098606,0.473526,0.163223,0.495313
2,Random Forest,0.89990,0.000000,0.000000,0.000000,0.494796



Confusion matrices (rows=true [0,1], cols=pred [0,1])

 Base (DT stump)
[[ 3518 14480]
 [  391  1611]]

 AdaBoost (DT stump)
[[9332 8666]
 [1054  948]]

 Random Forest
[[17998     0]
 [ 2002     0]]

Wrote: bonus_ensemble_resigned_table.tex
